# Embeddings des Évangiles — Mistral Batch

Exécuter les cellules étape par étape avec le kernel du projet. Ce notebook est livré non exécuté.

Dépendances : `uv pip install -r requirements-embeddings.txt` (dans le venv du projet).

Documentation : [Batch Mistral](https://docs.mistral.ai/studio/batch-processing).

In [1]:
import os
from getpass import getpass
from mistral_embeddings import (
    ROOT, prepare_jsonl, submit_batch, get_batch_job,
    save_embeddings, load_embeddings,
)

## Préparer les deux fichiers

Cet appel crée `data/batch/test.jsonl` et `gospels.jsonl`, ainsi que leurs métadonnées. Le test prend des versets répartis régulièrement dans les quatre Évangiles. `test_size` est configurable. Les fichiers de préparation existants sont remplacés : ne pas les régénérer avec d’autres paramètres pendant un batch en cours.

Le texte original est conservé dans `text`. Le champ `embedding_text` contient le texte envoyé, avec les marques ¶ retirées et les espaces normalisés.

In [2]:
files = prepare_jsonl(test_size=10)
files

{'test': {'jsonl': PosixPath('/Users/dinyad/Documents/IA/bible-analysis/data/batch/test.jsonl'),
  'metadata': PosixPath('/Users/dinyad/Documents/IA/bible-analysis/data/batch/test.metadata.json'),
  'count': 10},
 'gospels': {'jsonl': PosixPath('/Users/dinyad/Documents/IA/bible-analysis/data/batch/gospels.jsonl'),
  'metadata': PosixPath('/Users/dinyad/Documents/IA/bible-analysis/data/batch/gospels.metadata.json'),
  'count': 3779}}

## Choisir le fichier et soumettre

Garder `test` pour le premier essai. Pour le corpus complet, choisir `gospels`, puis exécuter les mêmes cellules. Chaque exécution de `submit_batch` crée un nouveau job. Les résultats sont enregistrés dans des dossiers distincts : `data/embeddings/test/` ou `data/embeddings/gospels/`.

In [3]:
from dotenv import load_dotenv

load_dotenv()

True

In [ ]:
dataset = "gospels"  # "gospels" pour tous les versets
selected = files[dataset]


In [5]:
key = os.environ.get("MISTRAL_API_KEY")
key

'mstrl_ptQLJAxdlkQyJW8pCL5q2kdekcFvJZqA_09ETLl'

In [ ]:
#job = submit_batch(selected["jsonl"])
#job_id = job["id"]
#print("Job ID à conserver :", job_id)

Job ID à conserver : c9a472bd-b58c-43d1-9df6-ef1787c27b58


## Consulter le statut

Réexécuter cette cellule pour consulter l’avancement, sans soumettre à nouveau. Après un redémarrage du kernel, réimporter les fonctions et renseigner le `job_id` conservé ; le fichier de métadonnées reste disponible sur disque.

In [39]:
#job_id = "e446ae64-2cde-49e1-ae83-83ee2efb8b41"
status = get_batch_job(job_id)
{k: status.get(k) for k in (
    "status", "total_requests", "succeeded_requests", "failed_requests", "error_file"
)}

{'status': 'SUCCESS',
 'total_requests': 3779,
 'succeeded_requests': 3779,
 'failed_requests': 0,
 'error_file': None}

## Télécharger et sauvegarder

À exécuter lorsque le statut est `SUCCESS`. La fonction refuse les résultats incomplets ou invalides. Elle associe les résultats par `custom_id` et restaure l’ordre du corpus.

- `embeddings.npy` : matrice float32, une ligne par verset, 1 024 colonnes.
- `metadata.json` : liste alignée, avec référence, texte original, texte envoyé, traduction, modèle et job.

Une nouvelle sauvegarde pour le même dataset remplace les résultats précédents. Pour conserver plusieurs essais, passer un `output_dir` distinct.

In [40]:
output_dir = save_embeddings(
    job_id,
    metadata_path=ROOT / "data/batch" / f"{dataset}.metadata.json",
)
output_dir

PosixPath('/Users/dinyad/Documents/IA/bible-analysis/data/embeddings/gospels')

## Charger pour l’analyse

Cette partie est locale et ne nécessite pas de clé API. `metadata[i]` décrit exactement `embeddings[i]`.

In [43]:
embeddings, metadata = load_embeddings(ROOT / "data/embeddings" / dataset)
print(embeddings.shape, embeddings.dtype)
metadata[0]

(3779, 1024) float32


{'custom_id': '40:1:1',
 'book_name': 'Matthieu',
 'book': 40,
 'chapter': 1,
 'verse': 1,
 'text': "¶ Généalogie de Jésus-Christ, fils de David, fils d'Abraham.",
 'embedding_text': "Généalogie de Jésus-Christ, fils de David, fils d'Abraham.",
 'translation': 'Louis Segond 1910',
 'row': 0,
 'model': 'mistral-embed',
 'job_id': 'c9a472bd-b58c-43d1-9df6-ef1787c27b58'}

In [44]:
embeddings[0]

array([-0.03170776,  0.00101185,  0.05654907, ..., -0.0173645 ,
        0.01154327,  0.00029659], shape=(1024,), dtype=float32)